# 机器学习与深度学习基础

> **本章定位**：复习后续章节共同依赖的机器学习、深度学习与训练闭环概念，建立统一的任务、数据、张量和评估术语。

> **章节边界**：本章只建立 MLP、CNN、RNN、Transformer 与 Diffusion 的方法坐标，不展开各架构的完整推导；BERT、ViT 与 Diffusion 分别见 `E30_nlp_bert.ipynb`、`E40_cv_vit.ipynb` 与 `E50_cv_diffusion.ipynb`。

> **总览**：内容从任务定义与数据隔离出发，依次覆盖类别 ID 与 One-hot 表示、经验风险、两层 MLP 的前向与反向传播、PyTorch 训练闭环、模型家族对照和生产交付边界。

```mermaid
flowchart LR
    A["任务与数据"] --> R["类别 ID / One-hot / 张量"]
    R --> B["模型"]
    B --> C["损失函数"]
    C --> D["参数优化"]
    D --> E["验证与选择"]
    E --> F["测试与交付"]
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 共同基础：基础复习 |
| 本章定位 | 复习机器学习、深度学习与 PyTorch 训练闭环，为后续数据、Tokenizer 与 Transformer 章节建立共同术语。 |
| 先修知识 | Python 函数、NumPy 数组、基本代数、损失函数与梯度下降。 |
| 预计时间 | 60～75 分钟 |
| 运行资源 | CPU 即可；无需下载模型权重。 |
| 输入 | 类别 ID 小型张量，以及具有 30 个数值特征的二分类数据。 |
| 交付物 | One-hot 原理实现与等价性证据、NumPy 两层 MLP、PyTorch 训练闭环、验证指标与运行配置记录。 |

### 1.1．学习目标

完成本章后，读者能够说明数据、模型、目标函数与评估指标如何共同定义学习任务，解释训练集、验证集与测试集的职责及隔离原则，根据张量形状描述前向传播、反向传播与参数更新过程，解释类别 ID、One-hot、Embedding 与交叉熵目标之间的关系，区分 logit、概率与决策阈值，解释 `BCEWithLogitsLoss` 的计算和数值稳定性，并区分常见模型架构的核心机制和适用范围。


In [ ]:
# 当前 Colab 环境通常已包含本章依赖；其他环境按仓库锁定版本安装。
# %pip install -r ../requirements.txt

## 2．直觉与输入输出契约

### 2.1．学习任务的组成

机器学习（Machine Learning，ML）从数据中估计映射 $f_\theta$，目标是在未参与参数拟合的数据上保持有效预测。

| 学习方式 | 数据形式 | 典型任务 | 优化目标 |
|---|---|---|---|
| 监督学习 | 特征 $X$ 与标签 $y$ | 分类、回归 | 预测已定义目标 |
| 无监督学习 | 只有 $X$ | 聚类、降维 | 发现数据结构 |
| 自监督学习 | 从数据自身构造监督信号 | 语言模型、表征学习 | 从未标注数据学习表示 |
| 强化学习 | 状态、动作、奖励 | 决策、模型对齐 | 最大化长期回报 |

本章的二分类输入为特征矩阵 $X\in\mathbb{R}^{N\times D}$，其中 $N$ 表示样本数、$D=30$ 表示特征数；标签 $y\in\{0,1\}^{N}$。模型先输出每个样本的 logit，再经 Sigmoid 得到形状为 $[N]$ 的正类概率，最后由冻结的阈值产生类别预测。

经验风险最小化写为：

$$
\theta^* = \arg\min_\theta \frac{1}{N}\sum_{i=1}^{N}\mathcal{L}(f_\theta(x_i), y_i) + \lambda\,\Omega(\theta)
$$

| 符号 | 含义 | 代码对象或形状 |
|---|---|---|
| $x_i$、$y_i$ | 第 $i$ 个样本及标签 | `features[i]` 为 $[D]$，`targets[i]` 为标量 |
| $f_\theta$ | 由参数 $\theta$ 定义的模型 | `baseline` 或两层 MLP |
| $\mathcal{L}$ | 单样本损失 | Logistic loss 或 BCE with logits |
| $\Omega(\theta)$ | 模型复杂度约束 | Logistic 正则项或 AdamW 权重衰减 |
| $\lambda$ | 正则权重 | 与 Logistic `C` 或 `weight_decay` 的配置相关 |

训练损失提供参数更新方向，评估指标用于模型选择与业务决策。训练集拟合参数，验证集选择配置，测试集仅用于方案冻结后的最终评估。


### 2.2．从类别 ID 到 One-hot

分类任务、Tokenizer 与离散特征都需要先建立稳定的类别到整数映射。设类别数为 $C$，某个类别 ID 为 $k\in\{0,\ldots,C-1\}$，One-hot（独热编码）向量 $\mathbf{o}\in\{0,1\}^{C}$ 定义为：

$$
o_j=\mathbb{1}[j=k]=\begin{cases}1,&j=k\\0,&j\ne k.\end{cases}
$$

| 符号或对象 | 含义 | 张量形状 |
|---|---|---|
| $C$ | 类别数或词表大小 | 标量 |
| $k$ | 单个类别 ID | 标量 |
| $\mathbf{o}$ | 单个 One-hot 向量 | $[C]$ |
| `class_ids` | 一批类别 ID | $[B]$ |
| `one_hot_vectors` | 一批 One-hot 向量 | $[B,C]$ |
| Token ID 序列 | 序列化离散 ID | $[B,L]$ |
| Token One-hot | 若显式展开词表轴 | $[B,L,V]$ |

One-hot 只表达“属于第 $k$ 类”，不表达类别间的距离或相似性。将 One-hot 与可学习矩阵 $E\in\mathbb{R}^{C\times D}$ 相乘，结果恰好是取出第 $k$ 行：

$$
\mathbf{o}^{\top}E=E_{k,:}.
$$

因此 `nn.Embedding(C, D)` 本质上是按 ID 查表，在数学上等价于 One-hot 乘 Embedding 矩阵，但生产实现不会物化巨大的稀疏 One-hot 张量。

| 表示 | 每个样本的约束 | 典型用途 |
|---|---|---|
| One-hot | 恰好一个元素为 1 | 互斥单标签的显式表示 |
| Multi-hot | 可有多个元素为 1 | 一个样本同时属于多个类别 |
| 概率或软标签 | 非负且通常和为 1 | Label Smoothing、知识蒸馏、不确定标注 |

类别 ID 的数值大小没有顺序含义。直接把 ID 当连续数值输入线性层，会虚构出“类别 3 大于类别 1”的度量关系；One-hot 或 Embedding 查表则保留离散身份。


### 2.3．传统机器学习训练与评估闭环

乳腺癌二分类基线采用分层划分：训练集用于拟合标准化器与逻辑回归，验证集用于观察候选配置，测试集用于冻结方案后的最终估计。标准化器封装在 `Pipeline` 中，因此均值与方差只从训练集获得。

原始数据包含 569 个样本和 30 个特征。按本章划分规则，预期形状为训练集 `[398, 30]`、验证集 `[85, 30]`、测试集 `[86, 30]`。评估函数输出 Accuracy、Precision、Recall、F1 与 ROC-AUC，所有指标均以同一正类定义和阈值计算。


In [ ]:
# 一个完整的传统机器学习基线：数据隔离、预处理、训练和评估。
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SPLIT_SEED = 42  # 固定样本归属；修改后必须重建划分并登记新的数据版本。
TRAINING_SEED = 42  # 固定初始化与训练 shuffle；质量比较需使用预先登记的多个种子。
HOLDOUT_FRACTION = 0.30  # 留出 30% 并等分，形成 70%/15%/15%；调整会改变拟合与评估样本量。
TEST_SHARE_OF_HOLDOUT = 0.50  # 验证集与测试集各占留出集的一半。
MALIGNANT_SOURCE_LABEL = 0  # 数据集固有标签：0 表示恶性。
LOGISTIC_REGULARIZATION_INVERSE = 1.0  # C 是正则强度的倒数；应按对数尺度在验证集选择。
LOGISTIC_MAX_ITERATIONS = 1_000  # 为标准化特征预留收敛空间；告警时还需复核尺度、容差与求解器。
CLASSIFICATION_THRESHOLD = 0.50  # 对称误判代价的起点；生产阈值应按验证集业务代价确定并冻结。

dataset = load_breast_cancer()
features = dataset.data.astype(np.float32, copy=False)
targets = (dataset.target == MALIGNANT_SOURCE_LABEL).astype(np.int64)

x_train, x_holdout, y_train, y_holdout = train_test_split(
    features, targets, test_size=HOLDOUT_FRACTION, random_state=SPLIT_SEED, stratify=targets
)
x_validation, x_test, y_validation, y_test = train_test_split(
    x_holdout,
    y_holdout,
    test_size=TEST_SHARE_OF_HOLDOUT,
    random_state=SPLIT_SEED,
    stratify=y_holdout,
)

baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        solver="newton-cholesky",  # 小型稠密二分类问题的确定性求解器，避免 SciPy L-BFGS 版本耦合。
        C=LOGISTIC_REGULARIZATION_INVERSE,
        max_iter=LOGISTIC_MAX_ITERATIONS,
    ),
)
baseline.fit(x_train, y_train)

def my_classification_metrics(model, features, targets):
    """计算二分类模型在给定特征与标签上的 AUC 和准确率，并返回两个标量指标。"""
    probabilities = model.predict_proba(features)[:, 1]
    predictions = (probabilities >= CLASSIFICATION_THRESHOLD).astype(np.int64)
    return {
        "accuracy": accuracy_score(targets, predictions),
        "precision": precision_score(targets, predictions, zero_division=0),
        "recall": recall_score(targets, predictions, zero_division=0),
        "f1": f1_score(targets, predictions, zero_division=0),
        "roc_auc": roc_auc_score(targets, probabilities),
    }

print("数据形状：", x_train.shape, x_validation.shape, x_test.shape)
print("验证集：", my_classification_metrics(baseline, x_validation, y_validation))
print("测试集：", my_classification_metrics(baseline, x_test, y_test))

<!-- theory-math-contract:v1 -->
### 2.4．核心机制的语言与数学表达

监督学习不是记住训练样本，而是在给定数据分布上寻找能够降低期望风险的参数。有限训练集只能给出经验风险，因此还需要独立验证集估计泛化误差：

$$
\hat{R}_{\mathrm{train}}(\theta)=\frac{1}{N}\sum_{i=1}^{N}\ell\!\left(f_\theta(x_i),y_i\right),\qquad
\theta_{t+1}=\theta_t-\eta\nabla_\theta\hat{R}_{\mathrm{train}}(\theta_t)
$$

其中，$x_i\in\mathbb{R}^{D}$ 是第 $i$ 个输入，$y_i$ 是目标，$f_\theta$ 是参数为 $\theta$ 的模型，$\ell$ 是单样本损失，$N$ 是训练样本数，$\eta$ 是学习率。代码中的 `loss` 对应 $\hat R$，`parameter.grad` 对应梯度；PyTorch 的优化器负责参数更新。经验风险下降只说明训练目标得到优化，不等价于验证风险或真实业务风险同步下降。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．深度学习训练闭环

深度学习（Deep Learning，DL）使用多层可微模块学习表示。两层神经网络的前向传播为：

$$
H = \phi(XW_1 + b_1), \qquad Z = HW_2 + b_2
$$

| 对象 | 含义 | 形状 |
|---|---|---|
| $X$ | 一批输入特征 | $[B,D]$ |
| $W_1$、$b_1$ | 第一层参数 | $[D,H]$、$[1,H]$ |
| $H$ | ReLU 隐藏表示 | $[B,H]$ |
| $W_2$、$b_2$ | 输出层参数 | $[H,1]$、$[1,1]$ |
| $Z$ | 二分类 logits | $[B,1]$ |

损失函数把整批预测压缩为标量。反向传播沿计算图应用链式法则，得到与每个参数形状一致的梯度；优化步骤再依据梯度更新参数。

```mermaid
flowchart LR
    X["输入 X [B,D]"] --> M["两层 MLP"]
    M --> Z["logits [B,1]"]
    Z --> L["标量损失"]
    Y["标签 y [B,1]"] --> L
    L --> G["参数梯度"]
    G --> O["参数更新"]
    O -.-> M
```

#### 3.1.1．`BCEWithLogitsLoss` 是什么

`BCEWithLogitsLoss` 用于**二分类**或**多标签分类**。它把 Sigmoid 和 Binary Cross-Entropy（BCE，二元交叉熵）合并成一个数值稳定的损失函数。模型输入给它的是未经 Sigmoid 的原始分数 $z$，称为 **logit**；logit 可以是任意实数，不是概率。

先从 logit 得到正类概率：

$$
p=\sigma(z)=\frac{1}{1+e^{-z}}
$$

再用真实标签 $y\in[0,1]$ 计算 BCE：

$$
\operatorname{BCE}(p,y)=-\left[y\log p+(1-y)\log(1-p)\right].
$$

`BCEWithLogitsLoss` 不会先显式计算可能被舍入为 `0` 或 `1` 的概率，而是使用与上式等价的稳定形式：

$$
\ell(z,y)=\max(z,0)-zy+\log\left(1+e^{-|z|}\right)
=\operatorname{softplus}(z)-zy.
$$

| logit $z$ | 概率 $\sigma(z)$ | $y=1$ 时的损失 | $y=0$ 时的损失 |
|---:|---:|---:|---:|
| `-2` | `0.119` | `2.127` | `0.127` |
| `0` | `0.500` | `0.693` | `0.693` |
| `2` | `0.881` | `0.127` | `2.127` |

因此，标签为 `1` 时更大的 logit 损失更小；标签为 `0` 时更小的 logit 损失更小。它对单个 logit 的梯度尤其直观：

$$
\frac{\partial \ell}{\partial z}=\sigma(z)-y.
$$

若 $y=1$ 而预测概率太低，梯度为负，优化器会推动 logit 增大；若 $y=0$ 而预测概率太高，梯度为正，优化器会推动 logit 减小。PyTorch 默认对 Batch 中所有元素的损失取平均值。

```python
# 训练：损失函数直接接收原始 logits，不能提前做 Sigmoid。
logits = model(features).squeeze(-1)
loss = nn.BCEWithLogitsLoss()(logits, targets.float())

# 推理：此时才把 logits 转成概率，再使用已冻结的阈值。
probabilities = torch.sigmoid(logits)
predictions = probabilities >= 0.5
```

输入 `logits` 与 `targets` 必须形状一致，例如都是 `[B]`；Target 使用浮点数且通常位于 `[0,1]`。不要写成 `BCEWithLogitsLoss(sigmoid(logits), targets)`，否则会重复执行 Sigmoid，改变损失和梯度。多标签分类可使用 `[B,C]`，每个类别独立计算 BCE；互斥的单标签多分类通常应使用 `CrossEntropyLoss`。类别不平衡时可基于训练集评估 `pos_weight`，但它会改变正类损失权重和概率校准，不能替代阈值选择与独立验证。


### 3.2．One-hot 原理实现与等价性

下面使用广播比较实现公式 $o_j=\mathbb{1}[j=k]$，不调用 `F.one_hot`。输入 `class_ids` 是任意形状的整数 ID 张量，输出在最后新增长度为 `num_classes` 的类别轴。实现后使用同一组数据验证四个不变量：

1. 每行只有一个 1，行和为 1。
2. `argmax` 可以恢复原始类别 ID。
3. 原理实现与 `torch.nn.functional.one_hot` 逐元素一致。
4. One-hot 矩阵乘 Embedding 表，与按 ID 查表结果一致。


In [ ]:
# 用四个类别建立可逐项核对的离散表示契约。
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

CLASS_NAMES = ["猫", "狗", "鸟", "鱼"]  # 固定类别顺序是数据 Schema 的一部分。
NUM_CLASSES = len(CLASS_NAMES)
class_ids = torch.tensor([2, 0, 3, 2], dtype=torch.long)  # 固定形状：[4]。

def my_one_hot(indices: torch.Tensor, num_classes: int) -> torch.Tensor:
    """按 o_j = 1[j=k] 将整数 ID 映射为最后一轴为类别的 float32 张量。"""
    if indices.dtype not in (torch.int8, torch.int16, torch.int32, torch.int64, torch.uint8):
        raise TypeError("One-hot 索引必须是整数张量")
    if num_classes <= 0:
        raise ValueError("num_classes 必须为正整数")
    if indices.numel() and (indices.min().item() < 0 or indices.max().item() >= num_classes):
        raise ValueError("One-hot 索引必须位于 [0, num_classes)")
    class_axis = torch.arange(num_classes, device=indices.device)
    return indices.unsqueeze(-1).eq(class_axis).to(torch.float32)

one_hot_vectors = my_one_hot(class_ids, NUM_CLASSES)  # 固定形状：[4, 4]。
library_one_hot = F.one_hot(class_ids, num_classes=NUM_CLASSES).to(torch.float32)
row_sum_error = float((one_hot_vectors.sum(dim=-1) - 1.0).abs().max())
round_trip_matches = bool(torch.equal(one_hot_vectors.argmax(dim=-1), class_ids))
library_matches = bool(torch.equal(one_hot_vectors, library_one_hot))

# 三维表只用于展示查表等价性；生产 Embedding 维度由模型容量与任务指标共同决定。
embedding_table = torch.tensor(
    [[0.2, 0.1, -0.3], [0.0, 0.4, 0.5], [-0.6, 0.2, 0.3], [0.7, -0.2, 0.1]],
    dtype=torch.float32,
)  # 固定形状：[4, 3]，轴语义为 [类别, 表示维度]。
embedded_by_matrix = one_hot_vectors @ embedding_table  # [4, 3]
embedded_by_lookup = embedding_table[class_ids]          # [4, 3]
embedding_layer = torch.nn.Embedding.from_pretrained(embedding_table.clone(), freeze=False)
embedded_by_library = embedding_layer(class_ids)         # [4, 3]
embedding_max_error = float(torch.stack([
    (embedded_by_matrix - embedded_by_lookup).abs().max(),
    (embedded_by_matrix - embedded_by_library).abs().max(),
]).max())

# 固定 logits 用于对照稀疏 ID 与 One-hot 两种交叉熵写法。
logits = torch.tensor(
    [[0.2, -0.1, 1.3, 0.0], [1.1, 0.4, -0.2, 0.1], [-0.5, 0.3, 0.2, 1.4], [0.1, -0.4, 0.9, 0.2]],
    dtype=torch.float32,
)  # 固定形状：[4, 4]，轴语义为 [样本, 类别]。
log_probabilities = torch.log_softmax(logits, dim=-1)
one_hot_cross_entropy = -(one_hot_vectors * log_probabilities).sum(dim=-1)
sparse_cross_entropy = F.cross_entropy(logits, class_ids, reduction="none")
probability_cross_entropy = F.cross_entropy(logits, one_hot_vectors, reduction="none")
cross_entropy_max_error = float(torch.stack([
    (one_hot_cross_entropy - sparse_cross_entropy).abs().max(),
    (one_hot_cross_entropy - probability_cross_entropy).abs().max(),
]).max())

if row_sum_error != 0.0 or not round_trip_matches or not library_matches:
    raise RuntimeError("One-hot 基本不变量验证失败")
if embedding_max_error > 1e-7 or cross_entropy_max_error > 1e-7:
    raise RuntimeError("One-hot 的 Embedding 或交叉熵等价性验证失败")

visual_matrix = one_hot_vectors.detach().float().cpu().numpy()
fig, axis = plt.subplots(figsize=(6.4, 3.6))
image = axis.imshow(visual_matrix, cmap="cividis", vmin=0.0, vmax=1.0, aspect="auto")
for row in range(visual_matrix.shape[0]):
    for column in range(visual_matrix.shape[1]):
        axis.text(column, row, f"{visual_matrix[row, column]:.0f}", ha="center", va="center", color="white")
axis.set(
    title="One-hot：类别 ID 展开为互斥类别轴",
    xlabel="类别轴", ylabel="样本",
)
axis.set_xticks(range(NUM_CLASSES), CLASS_NAMES)
axis.set_yticks(range(len(class_ids)), [f"sample {index}" for index in range(len(class_ids))])
fig.colorbar(image, ax=axis, ticks=[0, 1], label="指示值")
plt.tight_layout()
plt.show()

print({
    "shape": tuple(one_hot_vectors.shape),
    "row_sum_error": row_sum_error,
    "argmax_round_trip": round_trip_matches,
    "matches_F_one_hot": library_matches,
    "embedding_max_error": embedding_max_error,
    "cross_entropy_max_error": cross_entropy_max_error,
})


**应观察到的结论。** 矩阵每行只有一个高亮位置，其列索引与原始 ID 一致；数值摘要中行和误差、Embedding 最大误差和交叉熵最大误差应为 `0.0` 或浮点误差范围内的极小值，`argmax_round_trip` 和 `matches_F_one_hot` 应为 `True`。

**不可误读的边界。** 热力图中列与列之间的几何距离没有类别相似性含义；One-hot 只固化类别身份，相似性需由 Embedding 参数从数据中学习。小型四类矩阵用于验证机制，不表示大词表应显式展开为 One-hot。


### 3.3．两层 MLP 的 NumPy 实现

原理实现不依赖 `nn.Module`、自动微分或优化器，仅使用 NumPy 明确写出前向传播、反向传播和梯度下降：

$$
Z_1=XW_1+b_1,\qquad H=\operatorname{ReLU}(Z_1),\qquad Z_2=HW_2+b_2
$$

二分类采用数值稳定的 Binary Cross-Entropy with Logits（带 logits 的二元交叉熵）。反向传播从 $\partial\mathcal{L}/\partial Z_2$ 开始，依次计算 $W_2$、$H$、$Z_1$ 与 $W_1$ 的梯度。

隐藏宽度 `16` 使矩阵形状便于核对；全批量训练 `300` 步足以观察损失变化；学习率 `0.05` 适用于本章已标准化的小型数据。这组数值仅用于验证计算机制，不能直接视为生产训练配置。


In [ ]:
# 仅使用 NumPy 从零实现两层 MLP 的前向传播、反向传播和梯度下降。
from tqdm.auto import tqdm, trange
MANUAL_HIDDEN_FEATURES = 16  # 小型全批 MLP 的隐藏宽度；增大会提高容量与过拟合风险。
MANUAL_LEARNING_RATE = 0.05  # 当前规模的固定步长；震荡时降低，收敛缓慢时与训练步数联动复核。
MANUAL_TRAINING_STEPS = 300  # 足以展示损失下降；延长预算不能替代验证集与早停判断。
MANUAL_REPORT_EVERY_STEPS = 100

manual_scaler = StandardScaler()
manual_x_train = manual_scaler.fit_transform(x_train).astype(np.float64, copy=False)
manual_x_validation = manual_scaler.transform(x_validation).astype(np.float64, copy=False)
manual_y_train = y_train.astype(np.float64, copy=False).reshape(-1, 1)

rng = np.random.default_rng(TRAINING_SEED)
input_features = manual_x_train.shape[1]
# He 初始化让 ReLU 层的激活尺度保持稳定。
weight_1 = rng.normal(
    0.0, np.sqrt(2.0 / input_features), size=(input_features, MANUAL_HIDDEN_FEATURES)
)
# 固定形状：bias_1.shape = [1, 16]。
bias_1 = np.zeros((1, MANUAL_HIDDEN_FEATURES))
weight_2 = rng.normal(
    0.0, np.sqrt(2.0 / MANUAL_HIDDEN_FEATURES), size=(MANUAL_HIDDEN_FEATURES, 1)
)
# 固定形状：bias_2.shape = [1, 1]。
bias_2 = np.zeros((1, 1))

def my_sigmoid(values):
    # 分正负区间计算，避免 exp 在大幅负值处溢出。
    """以数值稳定的分段形式将任意形状的 logit 数组转换为同形状概率。"""
    probabilities = np.empty_like(values)
    nonnegative = values >= 0
    probabilities[nonnegative] = 1.0 / (1.0 + np.exp(-values[nonnegative]))
    negative_exp = np.exp(values[~nonnegative])
    probabilities[~nonnegative] = negative_exp / (1.0 + negative_exp)
    return probabilities

def my_forward(inputs):
    """执行两层 NumPy MLP 的前向传播，返回中间激活、logit 与 Sigmoid 概率。"""
    hidden_pre_activation = inputs @ weight_1 + bias_1
    hidden = np.maximum(hidden_pre_activation, 0.0)
    logits = hidden @ weight_2 + bias_2
    return hidden_pre_activation, hidden, logits

manual_losses = []
manual_gradient_norms = []
manual_progress = trange(
    MANUAL_TRAINING_STEPS, desc="训练 NumPy MLP", unit="step", dynamic_ncols=True
)
for step in manual_progress:
    hidden_pre_activation, hidden, logits = my_forward(manual_x_train)
    # logaddexp 给出数值稳定的 BCEWithLogits：log(1 + exp(z)) - y*z。
    loss = np.mean(np.logaddexp(0.0, logits) - manual_y_train * logits)

    logit_gradient = (my_sigmoid(logits) - manual_y_train) / manual_y_train.shape[0]
    weight_2_gradient = hidden.T @ logit_gradient
    bias_2_gradient = logit_gradient.sum(axis=0, keepdims=True)
    hidden_gradient = logit_gradient @ weight_2.T
    hidden_pre_activation_gradient = hidden_gradient * (hidden_pre_activation > 0.0)
    weight_1_gradient = manual_x_train.T @ hidden_pre_activation_gradient
    bias_1_gradient = hidden_pre_activation_gradient.sum(axis=0, keepdims=True)
    gradient_norm = np.sqrt(sum(
        np.square(gradient).sum()
        for gradient in (weight_1_gradient, bias_1_gradient, weight_2_gradient, bias_2_gradient)
    ))
    manual_losses.append(float(loss))
    manual_gradient_norms.append(float(gradient_norm))

    weight_1 -= MANUAL_LEARNING_RATE * weight_1_gradient
    bias_1 -= MANUAL_LEARNING_RATE * bias_1_gradient
    weight_2 -= MANUAL_LEARNING_RATE * weight_2_gradient
    bias_2 -= MANUAL_LEARNING_RATE * bias_2_gradient

    if (step + 1) % MANUAL_REPORT_EVERY_STEPS == 0 or step == 0:
        manual_progress.set_postfix(train_loss=f"{loss:.4f}")

_, _, validation_logits = my_forward(manual_x_validation)
validation_probabilities = my_sigmoid(validation_logits).reshape(-1)
validation_predictions = (validation_probabilities >= CLASSIFICATION_THRESHOLD).astype(np.int64)
print(
    "原理 MLP 验证集：",
    {
        "accuracy": accuracy_score(y_validation, validation_predictions),
        "recall": recall_score(y_validation, validation_predictions, zero_division=0),
        "f1": f1_score(y_validation, validation_predictions, zero_division=0),
    },
)

#### 3.3.1．损失与梯度范数的联合轨迹

学习问题是：反向传播产生的梯度如何与损失下降共同构成可观察的优化轨迹。下图直接使用上一单元每一步的 BCE Loss 与全局梯度范数，不另行拟合平滑曲线。验收条件是所有记录均为有限值，且固定配置下最终损失低于初始损失。


In [ ]:
# 用真实训练记录呈现标量损失与参数空间更新尺度。
import matplotlib.pyplot as plt

loss_trace = np.asarray(manual_losses, dtype=np.float64)
gradient_trace = np.asarray(manual_gradient_norms, dtype=np.float64)
if not np.isfinite(loss_trace).all() or not np.isfinite(gradient_trace).all():
    raise RuntimeError("损失或梯度范数包含非有限值")
if loss_trace[-1] >= loss_trace[0]:
    raise RuntimeError("固定配置下最终损失未低于初始损失")

steps = np.arange(1, len(loss_trace) + 1)
fig, loss_axis = plt.subplots(figsize=(9, 4.2))
gradient_axis = loss_axis.twinx()
loss_line = loss_axis.plot(steps, loss_trace, color="#0072B2", label="BCE Loss")[0]
gradient_line = gradient_axis.plot(
    steps, gradient_trace, color="#D55E00", linestyle="--", label="全局梯度范数"
)[0]
loss_axis.set(title="同一训练过程中的损失与梯度尺度", xlabel="参数更新步", ylabel="BCE Loss")
gradient_axis.set_ylabel("全局梯度 L2 范数")
loss_axis.grid(alpha=0.25)
loss_axis.legend([loss_line, gradient_line], [loss_line.get_label(), gradient_line.get_label()])
plt.show()
print({"initial_loss": loss_trace[0], "final_loss": loss_trace[-1], "max_gradient_norm": gradient_trace.max()})


损失整体下降说明当前梯度方向与学习率共同推动了经验风险降低；梯度范数反映当前参数空间中的更新信号尺度。梯度范数不要求单调下降，短时波动也不等于训练失败。单条训练曲线只验证这一固定数据划分、初始化与配置的机制，不能替代多 Seed、固定验证集和泛化评估。


## 4．证据验证

### 4.1．形状、数值与指标证据

NumPy 原理实现输出三类可观察证据：

1. 参数与中间张量保持约定形状：$W_1=[30,16]$、$H=[398,16]$、$W_2=[16,1]$、logits 为 $[398,1]$。
2. 每次报告的 BCE 损失均为有限标量；固定配置下，后期损失应低于初期损失，明显震荡或非有限值表示学习率、数据尺度或梯度计算存在问题。
3. 验证概率位于 $[0,1]$，预测数量与验证样本数一致，并输出 Accuracy、Recall 与 F1。

### 4.2．One-hot 的可证伪不变量

One-hot 单元同时输出结构、库对照与下游等价性证据：

- 输入 `[4]` 的类别 ID 得到 `[4,4]` 表示，每行和精确为 1，`argmax` 恢复原 ID。
- `my_one_hot` 与 `F.one_hot` 逐元素一致，说明公式和库语义对齐。
- `one_hot_vectors @ embedding_table`、`embedding_table[class_ids]` 与 `nn.Embedding(class_ids)` 一致，证明矩阵乘法与生产查表接口的数学等价性。
- 按公式计算 $-\sum_c y_c\log p_c$、稀疏 ID 目标的 `F.cross_entropy` 与 One-hot 概率目标的 `F.cross_entropy` 应给出相同的逐样本损失。

任一行和不为 1、ID 越界、Embedding 结果不一致或交叉熵差异超过浮点容差，都表示数据 Schema、类别轴或损失契约存在问题。

### 4.3．PyTorch 参考实现对照

随后使用同一数据边界建立 PyTorch 参考实现。两条路径的网络宽度和优化器并不相同，因此验收对象是数据隔离、张量方向、损失定义和指标口径的一致性，而不是逐参数或逐指标相等。

| 原理实现 | PyTorch 参考对象 |
|---|---|
| `X @ weight + bias` | `nn.Linear` |
| `np.maximum(z, 0)` | `nn.ReLU` |
| 稳定形式的 BCE with logits | `nn.BCEWithLogitsLoss` |
| 显式链式梯度 | `loss.backward()` |
| 梯度下降赋值 | `torch.optim.AdamW.step()` |
| 全批量数组 | `TensorDataset` 与 `DataLoader` |

参考实现应输出有限训练损失、验证指标和测试指标；测试结果只用于说明完整评估接口，不参与配置选择。


In [ ]:
# 同一分类任务的最小 PyTorch 实现。
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
HIDDEN_FEATURES = 32  # CPU 小网络的隐藏宽度；增大后需复核容量、延迟与过拟合。
BATCH_SIZE = 32  # 每次更新的样本数；改变 Batch、模型或精度时需联动复核学习率。
LEARNING_RATE = 1e-3  # AdamW 的初始学习率；应依据验证曲线和有效 Batch 重新选择。
WEIGHT_DECAY = 1e-4  # 轻量 L2 正则；取值变化需用固定划分比较泛化指标。
TRAINING_EPOCHS = 30  # 完整遍历训练集的上限；生产训练由验证指标与早停条件决定。
REPORT_EVERY_EPOCHS = 10

torch.manual_seed(TRAINING_SEED)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train).astype(np.float32, copy=False)
x_validation_scaled = scaler.transform(x_validation).astype(np.float32, copy=False)
x_test_scaled = scaler.transform(x_test).astype(np.float32, copy=False)

train_dataset = TensorDataset(
    torch.from_numpy(x_train_scaled), torch.from_numpy(y_train.astype(np.float32, copy=False))
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 库实现遵循 PyTorch 常用命名和组合方式；后续章节直接复用这种模式。
# 固定形状：model[2].weight.shape = [1, 32]（out_features, in_features）。
model = nn.Sequential(
    nn.Linear(x_train_scaled.shape[1], HIDDEN_FEATURES),
    nn.ReLU(),
    nn.Linear(HIDDEN_FEATURES, 1),
).to(DEVICE)
# 直接接收与 target 同形状的原始 logits；Sigmoid 已在损失内部稳定融合。
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999), eps=1e-8,
)

training_progress = tqdm(
    total=TRAINING_EPOCHS * len(train_loader), desc="训练 PyTorch MLP",
    unit="step", dynamic_ncols=True,
)
for epoch in range(TRAINING_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for batch_features, batch_targets in train_loader:
        batch_features = batch_features.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_features).squeeze(-1)
        loss = loss_function(logits, batch_targets)
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss.detach()) * batch_features.size(0)
        training_progress.update(1)

    mean_epoch_loss = epoch_loss / len(train_dataset)
    training_progress.set_postfix(
        epoch=f"{epoch + 1}/{TRAINING_EPOCHS}", train_loss=f"{mean_epoch_loss:.4f}"
    )
training_progress.close()

@torch.inference_mode()
def my_torch_metrics(model, features, targets):
    """在评估模式和无梯度上下文中计算 PyTorch 二分类模型的 AUC 与准确率。"""
    model.eval()
    feature_tensor = torch.from_numpy(features).to(DEVICE)
    probabilities = torch.sigmoid(model(feature_tensor).squeeze(-1)).cpu().numpy()
    predictions = (probabilities >= CLASSIFICATION_THRESHOLD).astype(np.int64)
    return {
        "accuracy": accuracy_score(targets, predictions),
        "recall": recall_score(targets, predictions, zero_division=0),
        "f1": f1_score(targets, predictions, zero_division=0),
        "roc_auc": roc_auc_score(targets, probabilities),
    }

print("运行设备：", DEVICE)
print("验证集：", my_torch_metrics(model, x_validation_scaled, y_validation))
print("测试集：", my_torch_metrics(model, x_test_scaled, y_test))

## 5．迁移到生产库

### 5.1．原理对象与 PyTorch 对象的映射

生产训练保留 `torch.nn` 的参数注册、自动微分、优化器、数据加载与设备迁移能力。NumPy 实现适合作为公式对应关系和小规模回归基线，不承担生产训练任务。迁移时需要显式处理以下差异：

- 原理实现采用全批量梯度下降，PyTorch 示例采用 mini-batch AdamW；两者的学习率含义和更新轨迹不同。
- `nn.BCEWithLogitsLoss` 接收 logits，不应在输入损失函数前重复执行 Sigmoid。
- `model.train()` 与 `model.eval()` 控制 Dropout、BatchNorm 等运行行为；推理阶段使用 `torch.inference_mode()` 避免构建反向图。
- `state_dict`、预处理器状态、阈值和依赖版本共同构成可重载制品。

### 5.2．从显式 One-hot 迁移到稀疏库接口

One-hot 是理解离散身份、查表和分类损失的公式桥梁，但生产路径应根据下游语义选择接口：

| 目标 | 原理对象 | PyTorch 接口 | 生产表示 |
|---|---|---|---|
| 核对 One-hot 语义 | `my_one_hot` | `F.one_hot` | 仅在调试、可视化或小类别特征中物化 |
| 将类别或 Token 变成稠密表示 | $\mathbf{o}^{\top}E$ | `nn.Embedding` | `torch.long` ID 查表，不展开 One-hot |
| 互斥单标签分类 | One-hot 交叉熵 | `nn.CrossEntropyLoss` | 默认使用 `[B]` 的整数类别 ID |
| 软标签或概率目标 | 概率加权交叉熵 | `nn.CrossEntropyLoss` | 使用与 logits 同形状的浮点概率目标 |
| 多标签分类 | Multi-hot 或独立二项目标 | `nn.BCEWithLogitsLoss` | `[B,C]` 浮点目标，每类独立 Sigmoid |

`F.one_hot` 输出整数张量；参与矩阵乘法或概率损失前需显式转为所需浮点精度。`CrossEntropyLoss` 的稀疏 ID 目标与概率目标形状不同，两者不能在数据管线中静默互换。

### 5.3．训练与泛化概念

| 概念 | 可观察现象 | 处理方向 |
|---|---|---|
| 欠拟合 | 训练集指标与验证集指标均较低 | 改进特征、模型容量或训练预算 |
| 过拟合 | 训练集持续改善而验证集退化 | 增加数据、正则化、Dropout 或早停 |
| 数据泄漏 | 训练过程使用了评估信息 | 按实体或时间划分，并只在训练集拟合预处理器 |
| 类别不平衡 | Accuracy 较高但少数类召回不足 | 同时评估 Precision、Recall、F1、PR-AUC，必要时采用重采样或代价敏感损失 |
| 梯度消失或爆炸 | 深层网络更新不稳定 | 使用合理初始化、归一化、残差连接与梯度裁剪 |
| 分布漂移 | 服务输入与训练数据分布发生变化 | 建立数据监控、回归评测与版本化再训练流程 |

损失函数负责优化，指标负责选择与决策，两者可以不同。测试集一旦参与反复调参，就不再提供独立的泛化估计。


### 5.4．常见模型架构与适用范围

| 架构 | 归纳偏置或核心机制 | 常见输入 | 典型用途 |
|---|---|---|---|
| 线性模型 / 树模型 | 简单边界、特征划分 | 表格数据 | 强基线、可解释预测 |
| MLP | 全连接非线性变换 | 固定长度向量 | 通用映射、分类头 |
| CNN | 局部连接与权重共享 | 图像、局部序列 | 视觉建模与局部模式识别 |
| RNN / LSTM | 递归状态 | 时序、文本 | 顺序建模 |
| Transformer | Attention 与并行序列建模 | Token 序列、图像 Patch | 大语言模型、BERT、ViT 与多模态模型 |
| Diffusion | 逐步去噪 | 图像或潜变量 | 图像、音频与视频生成 |

模型选择需要同时考察输入张量、参数化模块、训练目标、推理过程、数据规模与部署约束。后续章节从数据资产和 Tokenizer 开始，逐步展开 Transformer 及其训练与服务接口。


## 6．生产边界

本章示例用于建立训练闭环的共同语言。生产系统还需要落实以下边界：

- 类别名称到 ID 的映射顺序属于数据 Schema 和模型协议；增删、重排或复用 ID 会改变已有数据与权重的语义，需要版本化迁移和回归验证。
- 大词表 Token 序列若显式展开 One-hot，存储与计算会从 $O(BL)$ 增长为 $O(BLV)$；模型输入使用整数 ID 和 Embedding 查表，单标签交叉熵使用稀疏类别 ID。
- 训练 Schema 应明确区分互斥单标签、多标签与软标签，并固定 target 的形状、dtype、类别轴、缺失值和忽略值契约。
- 数据划分以用户、文档、会话或时间等真实泄漏单位为准，并将成员归属固化为数据版本。
- 预处理器只在训练集拟合；其状态与模型权重、类别定义、决策阈值和输入 Schema 一并发布。
- 验证集承担配置选择，测试集承担冻结方案的最终估计；发布后以独立回归集和线上监控跟踪分布漂移。
- 单次固定 seed 只支持同一环境下的调试与复核，不构成跨版本、跨设备的完全确定性保证；质量结论需要多个预先登记的训练 seed。
- 依赖版本、代码 revision、数据 revision、硬件、精度和关键指标应写入 Manifest，并提供模型与预处理器的重载检查。
- NumPy 原理实现仅保留为公式基线和回归参照；生产训练采用受维护的框架实现，并根据规模补充早停、Checkpoint、混合精度、分布式训练与监控。

后续章节均以类别 ID 与稀疏查表、样本与张量形状、数据隔离、logits 与概率、损失与指标，以及训练模式与推理模式的明确契约为前提。
